# Filtro 1

## Entreno de modelo

### Importación de librerías

In [9]:
import os
import numpy as np
import joblib
from deepface import DeepFace
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from tqdm import tqdm
import kagglehub

### Importción de dataset y definición de categorías

In [10]:
DATASET_ID  = "saramhai/people-with-and-without-glasses-dataset"
DOWNLOAD_PATH  = kagglehub.dataset_download(DATASET_ID)

CLASS_LABELS  = ["glasses", "no_glasses"] 
FACE_MODEL = "ArcFace"
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILE = "glasses_labels.pkl"

embeddings = []
labels = []


In [11]:
base_path = os.path.join(DOWNLOAD_PATH , "images")

for label_id, label_name in enumerate(CLASS_LABELS):
    label_path = os.path.join(base_path, label_name)
    if not os.path.exists(label_path):
        print(f"Advertencia: No se encontró la carpeta: {label_path}")
        continue

    img_files = [f for f in os.listdir(label_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    # Mezclar aleatoriamente
    np.random.shuffle(img_files)

    # Tomar 1/3 del total
    total_selected = len(img_files) // 3
    selected_files = img_files[:total_selected]

    print(f"\nProcesando categoría: {label_name} (Usando 1/3 = {total_selected} de {len(img_files)} imágenes)")

    # Extraer embeddings
    for filename in tqdm(selected_files, desc=label_name):
        file_path = os.path.join(label_path, filename)
        try:
            rep_data = DeepFace.represent(
                img_path=file_path,
                model_name=FACE_MODEL,
                enforce_detection=False,
                detector_backend='skip'
            )

            face_vector = rep_data[0]["embedding"]
            embeddings.append(face_vector)
            labels.append(label_id)

        except Exception as err:
            if 'face could not be detected' not in str(err):
                print(f"Error procesando {file_path}: {err}")

if not embeddings:
    print("Error: No se extrajo ningún embedding. Verifica el dataset.")
    exit()

embeddings = np.array(embeddings)
labels = np.array(labels)

print(f"\nTotal de embeddings extraídos: {embeddings.shape[0]}")
print(f"Dimensiones de cada embedding: {embeddings.shape[1]}")
print(f"Distribución de clases: {np.bincount(labels)}")

# Evaluación con train/test split
print("\nEvaluando modelo con split 70/30...")
X_train, X_test, y_train, y_test = train_test_split(
    embeddings, labels, test_size=0.3, random_state=42, stratify=labels
)

temp_model = SVC(kernel='linear', probability=True, random_state=42)
temp_model.fit(X_train, y_train)

y_pred = temp_model.predict(X_test)
print("\n" + classification_report(y_test, y_pred, target_names=CLASS_LABELS))

# Entrenamiento final con todos los datos
print("\nEntrenando modelo final con todos los datos...")
best_model = SVC(kernel='linear', probability=True, random_state=42)
best_model.fit(embeddings, labels)

# Guardar modelo y categorías
joblib.dump(best_model, MODEL_FILENAME)
joblib.dump(CLASS_LABELS, LABELS_FILE)

print(f"\nModelo: '{MODEL_FILENAME}'")
print(f"Categorías: '{LABELS_FILE}'")


Procesando categoría: glasses (Usando 1/3 = 923 de 2769 imágenes)


glasses: 100%|██████████| 923/923 [02:29<00:00,  6.19it/s]



Procesando categoría: no_glasses (Usando 1/3 = 717 de 2151 imágenes)


no_glasses: 100%|██████████| 717/717 [01:40<00:00,  7.16it/s]



Total de embeddings extraídos: 1640
Dimensiones de cada embedding: 512
Distribución de clases: [923 717]

Evaluando modelo con split 70/30...

              precision    recall  f1-score   support

     glasses       1.00      0.98      0.99       277
  no_glasses       0.98      1.00      0.99       215

    accuracy                           0.99       492
   macro avg       0.99      0.99      0.99       492
weighted avg       0.99      0.99      0.99       492


Entrenando modelo final con todos los datos...

Modelo: 'glasses_classifier.pkl'
Categorías: 'glasses_labels.pkl'


### Uso

In [2]:
import cv2
import numpy as np
import joblib
from deepface import DeepFace

# --- Configuración ---
MODEL_FILENAME = "glasses_classifier.pkl"
CATEGORIES_FILENAME = "glasses_labels.pkl"
FACE_MODEL = "ArcFace"  # ¡Debe ser el mismo que en el entrenamiento!

# --- Cargar modelos ---
try:
    model = joblib.load(MODEL_FILENAME)
    categories = joblib.load(CATEGORIES_FILENAME)
except FileNotFoundError:
    print(f"Error: No se encontraron los archivos '{MODEL_FILENAME}' o '{CATEGORIES_FILENAME}'.")
    print("Asegúrate de ejecutar el script de entrenamiento primero.")
    exit()

# Detector de rostros de OpenCV (para encontrar la cara RÁPIDO)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: No se pudo abrir la cámara.")
    exit()

print("Presiona 'q' para salir...")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Espejar la imagen para que parezca un espejo
    frame = cv2.flip(frame, 1)
    output_frame = frame.copy()
    
    # Usar el detector de OpenCV (rápido) para encontrar la cara
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(100, 100))

    for (x, labels, w, h) in faces:
        # Recortar el rostro (en BGR, DeepFace prefiere BGR)
        face_img = frame[labels:labels+h, x:x+w]

        # Evitar errores si el recorte está vacío
        if face_img.size == 0:
            continue

        try:
            # 1. Extraer embedding con DeepFace (el mismo modelo de entrenamiento)
            # Usamos 'sQQkip' porque ya le estamos pasando una cara recortada
            embedding_objs = DeepFace.represent(
                img_path=face_img,
                model_name=FACE_MODEL,
                enforce_detection=False, # No es necesario que DeepFace detecte de nuevo
                detector_backend='skip'  # ¡Importante!
            )
            
            # 2. Preparar el embedding para el modelo SVM
            embedding_vector = embedding_objs[0]["embedding"]
            embedding_data = np.array(embedding_vector).reshape(1, -1)

            # 3. Predecir con el modelo SVM
            prediction_idx = model.predict(embedding_data)[0]
            prediction_proba = model.predict_proba(embedding_data)[0]
            
            label = categories[prediction_idx]
            confidence = prediction_proba[prediction_idx] * 100

            # 4. Dibujar resultados
            color = (0, 255, 0) if label == "no_glasses" else (0, 0, 255)
            text = f"{label} ({confidence:.1f}%)"
            
            cv2.rectangle(output_frame, (x, labels), (x+w, labels+h), color, 2)
            cv2.putText(output_frame, text, (x, labels-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        except Exception as e:
            # DeepFace puede fallar si la cara es muy pequeña o está borrosa
            # print(f"Error de DeepFace: Qqqqq{e}") # Descomentar para depurar
            pass

    # Mostrar resultado
    cv2.imshow("Deteccion de Gafas (DeepFace + SVM)", output_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- Limpieza ---
cap.release()
cv2.destroyAllWindows()

Presiona 'q' para salir...


## Filtro

In [2]:
import cv2
import numpy as np
import dlib
import joblib
from deepface import DeepFace

# ----------------------
# CONFIG
# ----------------------
MODEL_FILENAME = "glasses_classifier.pkl"
LABELS_FILENAME = "glasses_labels.pkl"
FACE_MODEL = "ArcFace"

# Cargar clasificador entrenado
clf = joblib.load(MODEL_FILENAME)
class_labels = joblib.load(LABELS_FILENAME)

# Dlib detector y landmarks
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

# Colores para el tinte según giros
TINT_COLORS = [
    (0, 255, 255),   # Amarillo
    (255, 0, 255),   # Rosa
    (255, 255, 0),   # Cian
    (0, 128, 255),   # Naranja
    (128, 0, 255)    # Morado
]

def get_yaw(landmarks):
    # nariz izquierda (31) y derecha (35)
    left = landmarks[31]
    right = landmarks[35]
    nose = landmarks[30]

    dx = right[0] - left[0]
    dy = nose[0] - (left[0] + right[0]) / 2

    yaw = dy / dx
    return yaw

def extract_face_embedding(face_img):
    rep = DeepFace.represent(
        img_path=face_img,
        model_name=FACE_MODEL,
        enforce_detection=False,
        detector_backend="skip"
    )
    return rep[0]["embedding"]

# ----------------------
# LOOP PRINCIPAL
# ----------------------
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(frame_gray)

    for face in faces:
        x1, y1, x2, y2 = face.left(), face.top(), face.right(), face.bottom()

        # Recortar cara
        face_crop = frame[y1:y2, x1:x2]

        if face_crop.size == 0:
            continue

        # Extraer embedding
        try:
            emb = extract_face_embedding(face_crop)
        except:
            continue

        # Predicción
        pred = clf.predict([emb])[0]
        pred_label = class_labels[pred]

        # Obtener landmarks
        shape = predictor(frame_gray, face)
        landmarks = np.array([(shape.part(i).x, shape.part(i).y) for i in range(68)])

        # ----------------------
        # SI TIENE GAFAS → aplicar tinte
        # ----------------------
        if pred_label == "glasses":

            yaw = get_yaw(landmarks)
            yaw_index = int(abs(yaw) * 4) % len(TINT_COLORS)
            tint_color = TINT_COLORS[yaw_index]

            # Región de ojos (landmarks 36–41 y 42–47)
            left_eye_pts = landmarks[36:42]
            right_eye_pts = landmarks[42:48]

            # Función para llenar polígono
            def tint_region(pts):
                mask = np.zeros(frame.shape, dtype=np.uint8)
                cv2.fillConvexPoly(mask, pts, tint_color)
                return cv2.addWeighted(frame, 1, mask, 0.4, 0)

            # Aplicar tinte a cada cristal
            frame = tint_region(left_eye_pts)
            frame = tint_region(right_eye_pts)

            cv2.putText(frame, "GLASSES DETECTED", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        else:
            cv2.putText(frame, "NO GLASSES", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Filtro gafas", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
